# №10 — Ауыз су сапасын скринингтейтін гибридті жүйе

Бұл Google Colab жұмысы сараптамалық ережелерді, Kaggle деректерінде оқытылған модельді және түсіндірілетін шешім модулін біріктіреді. Жүйе су ішуге қауіпсіз деген сертификат бермейді: Kaggle жиынында микробиологиялық зерттеу мен барлық ластағыштар жоқ.

**Нұсқа:** №10. **Мақсат:** өлшемдерден Kaggle-дегі Potability класын болжау, жарамсыз кірісті тоқтату және күмәнді жағдайды маман тексеруіне жіберу.
**Қате құны:** суды қате түрде жарамды деу жалған сенім тудырады. Сондықтан шешім шегін Potable класының precision-ына сүйеніп таңдаймыз; шекке жақын не операциялық белгісі бар жағдайды тексеруге жібереміз.

## 1. Кітапханалар және деректер

Дереккөз: [Kaggle Water Potability](https://www.kaggle.com/datasets/adityakadiwal/water-potability). Kaggle бетінен датасетті ZIP немесе CSV түрінде жүктеп алыңыз. Келесі ұяшық екеуін де қабылдайды, ZIP ішіндегі CSV-ді өзі шығарып оқиды. Kaggle token орнатудың қажеті жоқ.

Кітапханалар: pandas/NumPy — деректер; scikit-learn — Pipeline, модельдер және метрикалар; Matplotlib/Seaborn — EDA; joblib — модельді сақтау; FastAPI/Pydantic/Uvicorn — API; Streamlit/requests — UI; pytest/httpx — тесттер.

Colab-та pandas, NumPy, scikit-learn, Matplotlib, Seaborn және joblib әдетте алдын ала орнатылған. Бұл notebook үшін қосымша кітапхана орнату қажет емес.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, is_zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

RANDOM_STATE = 42
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Бұрын жүктелген CSV-ді деректер бумасына жинаймыз.
for existing_csv in Path("/content").glob("*.csv"):
    (DATA_DIR / existing_csv.name).write_bytes(existing_csv.read_bytes())

# Егер бұрын CSV жүктелмесе, Kaggle ZIP не CSV файлын таңдаңыз.
csvs = sorted(DATA_DIR.rglob("*.csv"))
if not csvs:
    print("Kaggle-дан жүктеген ZIP архивін немесе water_potability.csv файлын таңдаңыз.")
    uploaded = files.upload()
    for name in uploaded:
        source = Path("/content") / name
        if source.suffix.lower() == ".csv":
            (DATA_DIR / source.name).write_bytes(source.read_bytes())
        elif is_zipfile(source):
            with ZipFile(source) as archive:
                for member in archive.namelist():
                    if member.lower().endswith(".csv"):
                        destination = DATA_DIR / Path(member).name
                        destination.write_bytes(archive.read(member))

csvs = sorted(DATA_DIR.rglob("*.csv"))
if not csvs:
    raise FileNotFoundError("CSV табылмады. Kaggle ZIP архивін немесе CSV файлын жүктеңіз.")
df = pd.read_csv(csvs[0])
print("Файл:", csvs[0].name, "| өлшемі:", df.shape)
display(df.head())

## 2. Деректерді түсіну

Kaggle файлында 9 сандық белгі және Potability нысаналы бағаны болуы керек. Бос мәндерді осы жерде толтырмаймыз: имputation Pipeline ішінде іске асады, сондықтан кросс-валидациядағы әр бүктеме медиананы тек training бөлігімен есептейді.

In [ ]:
TARGET="Potability"
FEATURES=["ph","Hardness","Solids","Chloramines","Sulfate","Conductivity","Organic_carbon","Trihalomethanes","Turbidity"]
missing=[c for c in FEATURES+[TARGET] if c not in df.columns]
if missing: raise ValueError(f"Күтілген бағандар жоқ: {missing}; CSV атауларын тексеріңіз.")
df=df[FEATURES+[TARGET]].copy()
for c in FEATURES+[TARGET]: df[c]=pd.to_numeric(df[c],errors="coerce")
df=df.dropna(subset=[TARGET]); df[TARGET]=df[TARGET].astype(int)
if not set(df[TARGET].unique()).issubset({0,1}): raise ValueError("Potability белгісі 0/1 болуы тиіс.")
print("Өлшем:",df.shape)
print("Типтер:")
display(df.dtypes)
print("Бос мәндер:")
display(df.isna().sum())
display(df.describe().T)
display(df[TARGET].value_counts().rename("саны").to_frame().assign(үлесі=df[TARGET].value_counts(normalize=True)))

## 3. EDA — алты график

Әр графикке қарап нақты қорытынды жасаңыз: класс үлесі, бос мәндер, үлестірім/шеткі мәндер, белгілердің байланысы. Қорытындыны төмендегі мәтін ұяшығына өз сөзіңізбен жазыңыз.

In [ ]:
sns.set_theme(style="whitegrid")
fig,ax=plt.subplots(figsize=(5,4)); df[TARGET].value_counts().sort_index().plot.bar(ax=ax,color=["#557da5","#d78c4a"]); ax.set(title="Potability кластары",xlabel="0 = жарамсыз, 1 = жарамды",ylabel="Үлгі саны"); plt.show()
fig,ax=plt.subplots(figsize=(8,4)); (df[FEATURES].isna().mean()*100).sort_values().plot.bar(ax=ax,color="#578a83"); ax.set(title="Бос мән үлесі",ylabel="Пайыз"); plt.show()
df[FEATURES].hist(figsize=(14,10),bins=30,layout=(3,3),color="#557da5"); plt.suptitle("Белгі үлестірімдері"); plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(14,5)); sns.boxplot(data=df[FEATURES],orient="h",ax=ax); ax.set_title("Шеткі мәндерге шолу (өлшем бірліктері әртүрлі)"); plt.show()
fig,ax=plt.subplots(figsize=(10,8)); sns.heatmap(df.corr(numeric_only=True),cmap="vlag",center=0,annot=True,fmt=".2f",ax=ax); ax.set_title("Корреляциялар"); plt.show()
fig,ax=plt.subplots(figsize=(7,5)); sns.scatterplot(data=df,x="ph",y="Turbidity",hue=TARGET,alpha=.45,ax=ax); ax.set_title("pH пен лайлылық және класс"); plt.show()

**EDA бойынша өз қорытындыңызды осында жазыңыз:**  
- Кластар теңгерімі қандай?  
- Бос мәні көп бағандар қайсы?  
- Қандай белгілерде қисық үлестірім/шеткі мән бар?  
- Күшті корреляция байқала ма?  
- Бір белгі классты жалғыз өзі ажырата ала ма?

Бақылау: датасет шағын; модель табылған байланыстарды ғана үйренеді. Бос мәнді имputation шынайы зертханалық өлшемнің орнын баспайды.

## 4. Сараптамалық ережелер және шектеулер

Үш топ: (1) физикалық тұрғыдан жарамсыз кіріс мәнін тоқтату; (2) жоқ өлшемді белгілеу, Pipeline медианамен толтыру; (3) модель оң болжаса да операциялық шектен ауытқуды маманға тексеруге жолдау. Төмендегі R02–R09 ережелері әр өлшенетін айнымалының теріс мәнін тексереді; R19–R27 бос мәнді белгілейді.

WHO pH 6.5–8.5-ийг ердийн ажиллагааны диапазон гэж сипаттайды, бірақ денсаулыққа негізделген шек қоймайды. WHO-ның 3 mg/L хлорамин мәні monochloramine үшін, Kaggle бағаны зат түрін нақтыламайды. Лайлылық 1 NTU — залалсыздандыру тиімділігінің нысаналы индикаторы. Бұлар автоматты қауіпсіздік сертификаты емес.

In [ ]:
rule_rows=[
("R01","pH 0–14 шегінен тыс","Қатаң дерек тексеру","Физикалық pH шкаласы"),
("R02–R09","Қалған 8 өлшемнің біреуі теріс","Қатаң дерек тексеру","Өлшем/концентрация теріс бола алмайды"),
("R19–R27","pH, қаттылық, solids, chloramines, sulfate, conductivity, organic carbon, THM не turbidity жоқ","Дерек ескертуі","Pipeline медианамен толтырады, нәтиже сақтықпен оқылады"),
("R14","pH 6.5–8.5-тен тыс","Кейінгі кеңес","WHO операциялық диапазоны; денсаулық шегі емес"),
("R15","Chloramines > 3 mg/L","Кейінгі кеңес","WHO monochloramine мәні; түр/бірлік тексеріледі"),
("R16","Turbidity > 1 NTU","Кейінгі кеңес","WHO бойынша залалсыздандыру тиімділігін тексеру"),
("R17","Ықтималдық threshold ±0.05 ішінде","Кейінгі кеңес","Маманның тексеруі қажет")]
pd.DataFrame(rule_rows,columns=["Rule ID","ЕГЕР","Тобы","Дерек/мақсат"])


Дереккөздер: [WHO Guidelines hub](https://www.who.int/teams/environment-climate-change-and-health/water-sanitation-and-health/water-safety-and-quality/drinking-water-quality-guidelines), [WHO pH and acceptability](https://www.who.int/teams/environment-climate-change-and-health/water-sanitation-and-health/water-safety-and-quality/drinking-water-quality-guidelines), [WHO chloramines fact sheet](https://cdn.who.int/media/docs/default-source/wash-documents/water-safety-and-quality/chemical-fact-sheets-2022/chloramines-fact-sheet-2022.pdf), [WHO small-supply operational targets](https://www.who.int/publications/i/item/9789240088740). Жергілікті санитарлық талапты қорғауда өз еліңіздің қолданыстағы ережесімен нақтылаңыз.

## 5. Модельдерді Pipeline ішінде салыстыру

Logistic Regression, Random Forest және HistGradientBoosting салыстырамыз. Әрқайсысына imputer Pipeline құрамында. Масштабтау логистикалық модельге ғана керек, сондықтан ол да Pipeline ішінде. CV метрикасы — average precision; әр класта стратификация сақталады.

In [ ]:
from sklearn.model_selection import train_test_split,StratifiedKFold,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier
from sklearn.metrics import precision_score,recall_score,f1_score,average_precision_score,classification_report,ConfusionMatrixDisplay,precision_recall_curve
from sklearn.inspection import permutation_importance
import joblib
X=df[FEATURES]; y=df[TARGET]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
X_fit,X_val,y_fit,y_val=train_test_split(X_train,y_train,test_size=.2,random_state=42,stratify=y_train)
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
candidates={
"LogisticRegression":(Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler()),("model",LogisticRegression(class_weight="balanced",max_iter=2000,random_state=42))]),{"model__C":[.1,1,10]}),
"RandomForest":(Pipeline([("imputer",SimpleImputer(strategy="median")),("model",RandomForestClassifier(class_weight="balanced",random_state=42,n_jobs=-1))]),{"model__n_estimators":[200,400],"model__max_depth":[None,8],"model__min_samples_leaf":[1,3]}),
"HistGradientBoosting":(Pipeline([("imputer",SimpleImputer(strategy="median")),("model",HistGradientBoostingClassifier(random_state=42))]),{"model__learning_rate":[.05,.1],"model__max_leaf_nodes":[15,31]})}
searches={}; rows=[]
for name,(pipe,grid) in candidates.items():
    search=GridSearchCV(pipe,grid,scoring="average_precision",cv=cv,n_jobs=-1,refit=True)
    search.fit(X_fit,y_fit); searches[name]=search
    rows.append({"model":name,"CV average precision":search.best_score_,"best params":search.best_params_})
cv_table=pd.DataFrame(rows).sort_values("CV average precision",ascending=False)
display(cv_table)
best_name=cv_table.iloc[0]["model"]
print("Таңдалған модель:",best_name)

## 6. Threshold баптау және тест бағалау

Potable=1 деп қате айту жалған сенім туғызуы мүмкін. Сондықтан training деректерінің validation бөлігінде precision кемінде 0.75 болатын threshold ішінен recall ең үлкенін аламыз. Егер ондай шек болмаса, validation F1 ең жоғарғы шегін таңдаймыз. Test бөлігі threshold таңдауға араласпайды.

In [ ]:
model=searches[best_name].best_estimator_
model.fit(X_fit,y_fit)
pval=model.predict_proba(X_val)[:,1]
prec,rec,th=precision_recall_curve(y_val,pval)
choices=[(float(p),float(r),float(t)) for p,r,t in zip(prec[:-1],rec[:-1],th) if p>=.75]
if choices:
    threshold=max(choices,key=lambda z:z[1])[2]; reason="precision >= .75 шектерінің ең жоғары recall-ы"
else:
    f1s=[2*p*r/(p+r) if p+r else 0 for p,r in zip(prec[:-1],rec[:-1])]
    threshold=float(th[int(np.argmax(f1s))]); reason="precision .75 шарты орындалмады; validation F1 максимумы"
model.fit(X_train,y_train)
ptest=model.predict_proba(X_test)[:,1]
pred=(ptest>=threshold).astype(int)
print("Threshold:",round(threshold,3),reason)
print(classification_report(y_test,pred,target_names=["not potable","potable"],zero_division=0))
print("Potable precision:",round(precision_score(y_test,pred,zero_division=0),3))
print("Potable recall:",round(recall_score(y_test,pred,zero_division=0),3))
print("Average precision:",round(average_precision_score(y_test,ptest),3))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=["not potable","potable"],cmap="Blues"); plt.show()

**Нәтижені талдау:** басым метрика таңдалған precision мақсатына жетті ме? Recall қанша болды? Нәтижеге қарай сенімділік туралы ғана қорытынды жасаңыз; тест жиыны мен Kaggle белгісіне қатысты белгісіздікті атап өтіңіз.

## 7. Белгі маңыздылығы және қате мысалдар

In [ ]:
perm=permutation_importance(model,X_test,y_test,n_repeats=10,random_state=42,scoring="average_precision",n_jobs=-1)
importance=pd.Series(perm.importances_mean,index=FEATURES).sort_values(ascending=False)
display(importance.to_frame("permutation importance"))
importance.sort_values().plot.barh(figsize=(7,5),title="Permutation importance"); plt.show()
err=X_test.copy(); err["actual"]=y_test; err["predicted"]=pred; err["p_potable"]=ptest
err=err[err.actual!=err.predicted].copy(); err["distance_to_threshold"]=(err.p_potable-threshold).abs()
display(err.sort_values("distance_to_threshold").head(5))

In [ ]:
artifact={"model":model,"threshold":float(threshold),"review_margin":0.05,"features":FEATURES,"medians":{k:float(v) for k,v in X_train.median().items()},"model_name":best_name,"threshold_reason":reason,
"metrics":{"precision_potable":float(precision_score(y_test,pred,zero_division=0)),"recall_potable":float(recall_score(y_test,pred,zero_division=0)),"average_precision":float(average_precision_score(y_test,ptest))}}
model_dir=Path("/content/water_hybrid_v10/models"); model_dir.mkdir(parents=True,exist_ok=True)
joblib.dump(artifact,model_dir/"model.joblib")
print("Модель сақталды:",model_dir/"model.joblib")

## 8. Ереже модулі және гибрид шешімнің үлгісі

Жоба бумасындағы src/rules.py және src/decision.py бұл логиканы модуль ретінде іске асырады. Төмендегі шағын демонстрация сол ойды көрсетеді: жарамсыз мәнді тоқтату, ықтималдықты есептеу, ал жақын/күмәнді жағдайда маманға жіберу.

In [ ]:
def hybrid_decision(probability, threshold, ph=None, chloramines=None, turbidity=None, margin=.05):
    fired=[]
    if ph is not None and not 6.5<=ph<=8.5: fired.append("R14: pH бойынша қосымша тексеру")
    if chloramines is not None and chloramines>3: fired.append("R15: monochloramine шегін зертханада нақтылау")
    if turbidity is not None and turbidity>1: fired.append("R16: залалсыздандыру тиімділігін тексеру")
    if abs(probability-threshold)<=margin: return "manual_review",fired+["R17: шешім шегіне жақын"]
    label="potable_screening" if probability>=threshold else "not_potable_screening"
    if label=="potable_screening" and fired: label="manual_review"
    return label,fired
print(hybrid_decision(float(ptest[0]),threshold,ph=7.2,chloramines=1,turbidity=.5))

## 9. API, Streamlit және pytest

Жоба бумасында FastAPI /health, /predict, /explain endpoint-тері бар; Pydantic кірісті тексереді. Streamlit интерфейсі API-ға HTTP сұрау жібереді, модельді тікелей шақырмайды. tests/test_core.py файлында 21 автоматты тест, оның ішінде теріс мән, NaN, қате тип, бос өріс, белгісіз өріс және шекке жақын болжау тексерістері бар.

Google Drive-қа экспорттаған соң модель файлын жоба бумасындағы models/model.joblib жолына қойыңыз. README.md ішінде орнату және іске қосу командалары көрсетілген.

## 10. Қорытындыны өз сөзіңізбен толтырыңыз

- Үш модельдің CV нәтижесі қалай өзгерді және қайсысын таңдадыңыз?
- Threshold мәні қандай? Precision пен recall-ға әсері?
- Қай белгілер көбірек әсер етті және 5 қате мысалда не байқалды?
- Қай ережелер қатаң тексереді, қайсысы маманға бағыттайды?
- Датасет нені қамтымайды және неге бұл жүйе зертхананы алмастырмайды?

**Дереккөздер:** [Kaggle dataset](https://www.kaggle.com/datasets/adityakadiwal/water-potability); [WHO drinking-water guidelines](https://www.who.int/teams/environment-climate-change-and-health/water-sanitation-and-health/water-safety-and-quality/drinking-water-quality-guidelines); [WHO chloramines fact sheet](https://cdn.who.int/media/docs/default-source/wash-documents/water-safety-and-quality/chemical-fact-sheets-2022/chloramines-fact-sheet-2022.pdf); [WHO small water supplies](https://www.who.int/publications/i/item/9789240088740).